In [1]:
import os
import time
from PIL import Image
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ==========================================
# 1. CẤU HÌNH PIPELINE & MÔ PHỎNG VRAM
# ==========================================
# Thay thế bằng đường dẫn tới thư mục model Qwen-VL của bạn
MODEL_NAME = "/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct" 

print("[INFO] Đang khởi tạo Tokenizer và vLLM Engine...")

# Khởi tạo Tokenizer để dựng Chat Template chuẩn của dòng Qwen
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Khởi tạo vLLM - Giữ mức 0.55 để giả lập GPU 16GB như bạn đang cấu hình
llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    gpu_memory_utilization=0.55,  # Mức mô phỏng bộ nhớ card 16GB
    max_model_len=4096            # Đảm bảo đủ chỗ cho cả token hình ảnh lớn
)

# Tham số lấy mẫu tối ưu cho OCR (Temperature = 0.0 để cho ra kết quả chính xác nhất)
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1024  # Cho phép sinh đoạn văn bản dài nếu ảnh nhiều chữ
)


# ==========================================
# 2. ĐỊNH NGHĨA PROMPT & DANH SÁCH ẢNH LOCAL
# ==========================================
def build_ocr_prompt(tokenizer):
    """Tạo prompt hệ thống ép model hoạt động như một máy quét OCR thuần túy"""
    messages = [
        {
            "role": "system",
            "content": "You are a precise OCR engine. Your task is to extract and transcribe all visible text from the image exactly as it appears. Maintain the layout/lines if possible. Do not add introductory or conversational text."
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Please perform OCR on this image and transcribe all text."}
            ]
        }
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ⚠️ HÃY THAY THẾ CÁC ĐƯỜNG DẪN DƯỚI ĐÂY THÀNH ĐƯỜNG DẪN ĐẾN FILE ẢNH CỦA BẠN
image_paths = [
    "/home/drnguyenvinh/Exam-Assistant/Bench_mark/image_test/account_test.png",
    "/home/drnguyenvinh/Exam-Assistant/Bench_mark/image_test/Exam_code_test.png",
    "/home/drnguyenvinh/Exam-Assistant/Bench_mark/image_test/login_test.png",
    "/home/drnguyenvinh/Exam-Assistant/Bench_mark/image_test/test_new.png",
    "/home/drnguyenvinh/Exam-Assistant/Bench_mark/image_test/test.png"
]


# ==========================================
# 3. TIẾN HÀNH XỬ LÝ VÀ CHẠY OCR INFERENCE
# ==========================================
vllm_inputs = []
valid_paths = []

# Đọc ảnh và đóng gói dữ liệu đầu vào
for path in image_paths:
    if not os.path.exists(path):
        print(f"[🚨 WARNING] Không tìm thấy file tại đường dẫn: '{path}' -> Bỏ qua.")
        continue
    
    try:
        # Load ảnh thông qua PIL và đưa về hệ màu RGB chuẩn
        image = Image.open(path).convert("RGB")
        prompt = build_ocr_prompt(tokenizer)
        
        vllm_inputs.append({
            "prompt": prompt,
            "multi_modal_data": {"image": image}
        })
        valid_paths.append(path)
    except Exception as e:
        print(f"[ERROR] Lỗi khi đọc file {path}: {e}")

# Tiến hành Inference hàng loạt (Batch Inference) nếu có ảnh hợp lệ
if vllm_inputs:
    print(f"\n[INFO] Bắt đầu OCR cho {len(vllm_inputs)} hình ảnh đồng thời...")
    start_time = time.time()
    
    # vLLM tự động phân phối xử lý song song các ảnh này trên GPU
    outputs = llm.generate(vllm_inputs, sampling_params=sampling_params)
    
    total_time = time.time() - start_time
    print(f"[SUCCESS] Hoàn thành OCR! Tổng thời gian xử lý: {total_time:.2f} giây.\n")
    
    # In kết quả hiển thị ra màn hình Terminal/Notebook
    for path, output in zip(valid_paths, outputs):
        ocr_result = output.outputs[0].text
        print("\n" + "="*70)
        print(f"📷 KẾT QUẢ OCR CHO FILE: {os.path.basename(path)}")
        print("="*70)
        print(ocr_result.strip())
        print("="*70)
else:
    print("[🚨 ERROR] Không tìm thấy hoặc không có hình ảnh nào hợp lệ để chạy pipeline.")

/opt/miniconda3/envs/qwen3_bm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Đang khởi tạo Tokenizer và vLLM Engine...
INFO 07-03 21:45:56 [utils.py:240] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.55, 'disable_log_stats': True, 'model': '/home/drnguyenvinh/Exam-Assistant/Bench_mark/Qwen3-VL-4B-Instruct'}
INFO 07-03 21:45:56 [model.py:568] Resolved architecture: Qwen3VLForConditionalGeneration
INFO 07-03 21:45:56 [model.py:1697] Using max model len 4096


2026-07-03 21:45:56,810	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 07-03 21:45:56 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-03 21:45:56 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 07-03 21:45:56 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


...

(EngineCore pid=3437284) INFO 07-03 21:46:29 [gpu_model_runner.py:6243] Graph capturing finished in 3 secs, took 0.57 GiB
(EngineCore pid=3437284) INFO 07-03 21:46:29 [gpu_worker.py:621] CUDA graph pool memory: 0.57 GiB (actual), 0.51 GiB (estimated), difference: 0.05 GiB (9.5%).
(EngineCore pid=3437284) INFO 07-03 21:46:29 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=3437284) INFO 07-03 21:46:29 [core.py:299] init engine (profile, create kv cache, warmup model) took 27.56 s (compilation: 17.11 s)
(EngineCore pid=3437284) INFO 07-03 21:46:29 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])

[INFO] Bắt đầu OCR cho 5 hình ảnh đồng thời...


Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(EngineCore pid=3437284) WARNING 07-03 21:46:34 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=3437284) WARNING 07-03 21:46:34 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _bilinear_pos_embed_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=3437284) WARNING 07-03 21:46:34 [jit_monitor.py:103] Triton kernel JIT compilation during inference: rotary_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 5/5 [00:01<00:00,  2.74it/s, est. speed input: 753.02 toks/s, output: 130.44 toks/s]

[SUCCESS] Hoàn thành OCR! Tổng thời gian xử lý: 6.22 giây.


📷 KẾT QUẢ OCR CHO FILE: account_test.png
[EOS Client 24.02.20.16] Please Login...
Exam Code: ENT104RE_LS16_P1
User N
Pass
Do
Exam Registering
× The account is NOT allow to take the exam!
OK
Register the exam may take some minutes, please wait

📷 KẾT QUẢ OCR CHO FILE: Exam_code_test.png
Start exam
Exam code is not available!
OK

📷 KẾT QUẢ OCR CHO FILE: login_test.png
Login failed
Sorry, unable to verify your information. Check [User Name] and [Password]!
OK

📷 KẾT QUẢ OCR CHO FILE: test_new.png
VietNTCE19
EOS Login Form
You cannot take the exam if EOS runs under a virtual machine
Exam Code: DRP101.SP24.J2_PT_54273
User Name:
Password:
Domain:
21.07.2020
Connecting...
Start Exam Error
Cannot connect to the EOS server
Check sound (12s)
OK
Register the exam may take time, please wait!
GMT%8477

📷 KẾT QUẢ OCR CHO FILE: test.png
Connecting...
Start Exam Error: Connect to the server [10.22.198.100:29697] failure!
OK
